In [1]:
suppressPackageStartupMessages({
    require(circlize)
    library(dplyr)
})

In [2]:
mat  <- read.delim("../1.mapping/Hypo_Mmus_Bflo.MappingTables.links.csv", header = T, sep = ",")

In [3]:
mat$groups1 <- vapply(mat$source, FUN = function(x){strsplit(x, split = '_')[[1]][2]}, FUN.VALUE = character(1))
mat$groups2 <- vapply(mat$target, FUN = function(x){strsplit(x, split = '_')[[1]][2]}, FUN.VALUE = character(1))

In [4]:
# keep only neuronal clusters
n <- c('C25-1: GLU-1', 'C25-2: GLU-2','C25-3: GLU-3','C25-4: GLU-4','C25-5: GLU-5','C25-6: GLU-6','C25-7: GLU-7','C25-8: GLU-8','C25-9: GLU-9',
       'C25-11: GABA-1','C25-10: GABA-2','C25-12: GABA-3','C25-13: GABA-4','C25-14: GABA-5', 'C25-23: ParsTuber',
       18,11,10,13,7,15,0,1,14,21,2,17,16,9)

mat <- mat %>% filter(groups1 %in% n) %>% filter(groups2 %in% n)
mat <- mat %>% mutate(groups1 = factor(groups1, levels = rev(n)), 
                     groups2 = factor(groups2, levels = rev(n))) %>%
  arrange(groups1)

In [5]:
# manually set colours for clusters
color = setNames(n, c(rep('#EEBEC0', times = 9), rep('#23B2E0', times = 5), '#F26B8A',
                      "#0072B2", "#E69F00", "#56B4E9", "#56B4E9", "#009E73", "#D55E00","#F0E442","#F0E442",
                      "#CC79A7", "#787878", "#663300", "#663300", "#008080", "#008080"
                     ))
color

#EEBEC0             #EEBEC0             #EEBEC0             #EEBEC0 
     "C25-1: GLU-1"      "C25-2: GLU-2"      "C25-3: GLU-3"      "C25-4: GLU-4" 
            #EEBEC0             #EEBEC0             #EEBEC0             #EEBEC0 
     "C25-5: GLU-5"      "C25-6: GLU-6"      "C25-7: GLU-7"      "C25-8: GLU-8" 
            #EEBEC0             #23B2E0             #23B2E0             #23B2E0 
     "C25-9: GLU-9"    "C25-11: GABA-1"    "C25-10: GABA-2"    "C25-12: GABA-3" 
            #23B2E0             #23B2E0             #F26B8A             #0072B2 
   "C25-13: GABA-4"    "C25-14: GABA-5" "C25-23: ParsTuber"                "18" 
            #E69F00             #56B4E9             #56B4E9             #009E73 
               "11"                "10"                "13"                 "7" 
            #D55E00             #F0E442             #F0E442             #CC79A7 
               "15"                 "0"                 "1"                "14" 
            #787878             #663300             #663300             #008080 
               "21"                 "2"                "17"                "16" 
            #008080 
                "9"

In [6]:
# assign colours to the mapping table
mat$color <- names(color)[match(mat$groups1, color)]
mat$sp <- vapply(mat$source, FUN = function(x){
    strsplit(x, split = '_')[[1]][1]
}, FUN.VALUE = character(1))
mat$sp2 <- vapply(mat$target, FUN = function(x){
    strsplit(x, split = '_')[[1]][1]
}, FUN.VALUE = character(1))

In [7]:
# select linkages between amphioxus and vertebrates
mat <- mat %>% filter(sp == 'bf' | sp2 == 'bf')
# show only one direction
mat[mat$sp2 == 'bf', 'value'] = 0

In [8]:
# manually add gaps
test = vapply(unique(mat[[1]]), FUN = function(x){strsplit(x, split = '_')[[1]][1]},FUN.VALUE = character(1))
gaps = numeric(length(test))
for (i in 1:(length(gaps)-1)) {
  # Check if the current element is different from the previous one
  if (test[i] != test[i + 1]) {
    gaps[i] <- 1
  } else {
    gaps[i] <- 0
  }
}

In [9]:
circos.par(gap.after = gaps, track.margin = c(0, 0.01))
grid.col = setNames(mat$color, mat$source)
border_df = mat[grepl("bf", mat$source), 1:2]
border_df = rbind(border_df, mat[grepl("bf", mat$target), 1:2])
#border_df <- rbind(border_df, border_df2)
border_df$p = rep(x = 1, times = nrow(border_df))

In [10]:
pdf("Bf_vs_Mm_Hypo_neurons.chord_plot.pdf", width = 20, height = 20)

species_colors <- c("hs" = "#989A9C", "mm" = "#F7D08D", "pv" = "#BF83A5", "pm" = "#8684B0", "bf" = '#91C79D')
chordDiagram(mat[,1:3], directional = 1, direction.type = c("arrows"), link.arr.type = "big.arrow", 
            grid.col = grid.col, 
            link.visible = mat[[3]] > 0.15,
            annotationTrack = "grid", preAllocateTracks = list(track.height = 0.02))

circos.trackPlotRegion(
    track.index = 1,
  ylim = c(0, 1),  # Define the vertical range for the track
  track.height = 0.02,  # Adjust track height as needed
  bg.border = NA,  # No border for the track
  panel.fun = function(x, y) {
    sector.index <- CELL_META$sector.index  # Current sector
    species <- mat$sp[mat$source == sector.index]  # Match species to the current sector
    
    # Draw colored rectangles for each species
    circos.rect(
      xleft = CELL_META$cell.xlim[1], xright = CELL_META$cell.xlim[2], 
      ybottom = 0, ytop = 1, 
      col = species_colors[species], border = NA
    )
  }
)
circos.track(track.index = 2, panel.fun = function(x, y) {
    circos.text(CELL_META$xcenter, CELL_META$ylim[1], CELL_META$sector.index, 
        facing = "clockwise", niceFacing = TRUE, adj = c(0, 0.05), cex = 1)
}, bg.border = NA) # here set bg.border to NA is important


dev.off()
circos.clear()

pdf 
  2